In [24]:
from langchain.chat_models import init_chat_model
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.messages import HumanMessage
from langchain.messages import AIMessage
from pprint import pprint
from langchain_core.chat_history import (BaseChatMessageHistory, InMemoryChatMessageHistory)

In [6]:
print("""
        Building a chatbot   
        In this video we'll go over an example of how to design and implement an llm-powered chatbot
        This chatbot will be able to have a conversation and remember previous interacions
        
        Note that this chatbot that we build will only use the language model to have a conversation. There are several other related conecpts that you may be looking for
        
        > Conversational RAG: Enable a chatbot experience over an exteernal source of data
        > Agents: Build a chatbot that can take actions
        
      """)


        Building a chatbot   
        In this video we'll go over an example of how to design and implement an llm-powered chatbot
        This chatbot will be able to have a conversation and remember previous interacions

        Note that this chatbot that we build will only use the language model to have a conversation. There are several other related conecpts that you may be looking for

        > Conversational RAG: Enable a chatbot experience over an exteernal source of data
        > Agents: Build a chatbot that can take actions

      


In [7]:
model = init_chat_model(
    model_provider = 'groq',
    model = 'openai/gpt-oss-20b',
    temperature = 0.50
)

In [12]:
## HumanMessage -> indicando que foi um user que mandou a mensagem
## AiMessage -> objeto retornado pela llm, content é o conteudo em si desse objeto
resultado1 = model.invoke([
    HumanMessage(content="Hi, my name is Mateus and I'm a AI agent engineer")
])

print(resultado1.content)

Hello Mateus! 👋 It’s great to meet an AI agent engineer. How can I assist you today? Whether it’s a technical question, brainstorming ideas, or just a chat about AI, I’m here to help.


In [16]:
### Modo manual (Stateless)
### aparentemente, nesse contexto, a llm teve uma memoria de lembrar quem eu sou e oque eu faço
### porem, todos os message types foram enviados de uma vez para a janela de contexto da llm
### quando esse chat finalizar, ele não vai lembrar de nada
resultado2 = model.invoke([
    HumanMessage(content="Hi, my name is Mateus and I'm a AI agent engineer"),
    AIMessage(content="Hello Mateus! 👋 It’s great to meet an AI agent engineer. How can I assist you today? Whether it’s a technical question, brainstorming ideas, or just a chat about AI, I’m here to help."),
    HumanMessage(content="Hey, whats my name and what do I do?")
])

print(resultado2.content)

You’re Mateus, and you’re an AI agent engineer—essentially a developer who builds, fine‑tunes, and maintains AI agents that can interact, reason, and perform tasks across various domains.


In [15]:
print("""
      Message History

      We can use a message history class to wrap our model and make it stateful.
      This will keep track of inputs and outputs of the model, and store them in some datastore
      Future interactions will then load those messages and pass then into the chain as part of the input
      
      Resume:
      
      1) Message History Class
         Em vez de ter que gerenciar manualmente uma lista de mensagens como o HumanMessages, AIMessages..... 
         o langchain oferece uma classe utilitária como o RunnableWithMessageHistory que envolve o meu modelo 
         
      2) Deixando o modelo Stateful (Com estado):
        Modelos de llm por padrão, não tem memoria. Essa classe atua como um intermediário que intercepta as entradas e saidas e salva automaticamente a conversa em um banco de dados ou memoria persistente como redis, SQLite, na ram local... 

      3) Injeção automática do historico
        Nas interações seguintes, eu envio somente uma nova pergunta, a classe de historico recupera automaticamente as mensgens anteriores salvas no banco de dados, monta a lista inteira e injeta na llm junto com a pergunta atual
      """)


      Message History

      We can use a message history class to wrap our model and make it stateful.
      This will keep track of inputs and outputs of the model, and store them in some datastore
      Future interactions will then load those messages and pass then into the chain as part of the input

      Resume:

      1) Message History Class
         Em vez de ter que gerenciar manualmente uma lista de mensagens como o HumanMessages, AIMessages..... 
         o langchain oferece uma classe utilitária como o RunnableWithMessageHistory que envolve o meu modelo 

      2) Deixando o modelo Stateful (Com estado):
        Modelos de llm por padrão, não tem memoria. Essa classe atua como um intermediário que intercepta as entradas e saidas e salva automaticamente a conversa em um banco de dados ou memoria persistente como redis, SQLite, na ram local... 

      3) Injeção automática do historico
        Nas interações seguintes, eu envio somente uma nova pergunta, a classe de histor

In [ ]:
### sempre que diferentes usuarios estiverem conversando com a llm, como vamos garantir que uma sessão seja diferente da outra?
### temos que diferenciar a sessão, pois, cada user tem seu id unico, para não misturar conversas



model = init_chat_model(
    model_provider = 'groq',
    model = 'openai/gpt-oss-20b',
    temperature = 0.50
)


"""
Store:: 
 -> funciona como um banco de dados para a troca de mensagens tanto da parte HumanMessage quanto IAMessage
 -> Ele vai guardar cada conversa usando o sesion_id como chave
 
 
 -> Debuguei ela nas proximas celulas
"""
store = {}



"""
-> O langchain precisa de uma forma de saber aonde buscar ou salvar o historico toda vez que uma mensagem nova chega, essa função faz exatamente isso

Logica:
    if session_id not in store -> verifica se é a primeira vez que a sessão esta enviando mensagem
    se session_id não existir no dicionario :: ele cria uma nova instancia limpa de InMemoryChatMessageHistory() e salva dentro do store
    devolve o objeto de historico correspondente a aquele session_id
"""

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


"""
-> É como se fosse o orquestrador do fluxo
-> É ele que intercpta as chamadas de entrada e saida de respostas e injeta no nosso banco de dados 'store'

"""

with_message_history = RunnableWithMessageHistory(
    model,
    get_session_history
)

### Define a configuração da sessão
config = {"configurable": {"session_id": "usuario_mateus_123"}}

### Primeira interação: envia apenas a nova mensagem
res1 = with_message_history.invoke(
    [HumanMessage(content="Boa tarde, meu nome é Mateus e estou estudando pra me tornar um engenheiro de agentes de ia")],
    config=config
)
print("Resposta 1:", res1.content)

print("-" * 50)


c:\Users\Mateus\Desktop\Udemy\Complete Agentic AI Bootcamp\Seção 7 - Building Basic LLM Application using LCEL\AmbienteSecao7\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Resposta 1: Boa tarde, Mateus! 👋

Que bacana saber que você está se preparando para se tornar um engenheiro de agentes de IA. Esse campo está crescendo rapidamente e oferece muitas oportunidades interessantes.

### Como posso ajudar você hoje?

1. **Fundamentos de IA e Machine Learning**  
   - Conceitos básicos, algoritmos, métricas, etc.

2. **Arquiteturas de Agentes**  
   - Agentes reativos, deliberativos, híbridos, multi‑agentes, etc.

3. **Frameworks e Ferramentas**  
   - OpenAI, LangChain, Rasa, Microsoft Bot Framework, etc.

4. **Desenvolvimento de Conversational Agents**  
   - NLP, LLMs, fine‑tuning, prompt engineering, avaliação de diálogos.

5. **Ética e Responsabilidade**  
   - Bias, privacidade, segurança, governança de IA.

6. **Projetos Práticos**  
   - Ideias de projetos, tutoriais, boas práticas de codificação.

7. **Carreira e Networking**  
   - Como montar um portfólio, participar de comunidades, encontrar oportunidades de trabalho.

8. **Outros tópicos**  
   -

In [22]:
# Segunda interação: envia apenas a pergunta atual.
# O LangChain resgata automaticamente o histórico salvo na sessão "usuario_mateus_123"
res2 = with_message_history.invoke(
    [HumanMessage(content="Olá, qual é o meu nome e oque eu faço profissionalmente?")],
    config=config
)
print("Resposta 2:", res2.content)

Resposta 2: Seu nome é **Mateus**.  
Profissionalmente, você está **estudando para se tornar engenheiro de agentes de IA** – ou seja, você está se preparando para projetar, desenvolver e manter sistemas inteligentes que interagem de forma autônoma ou semi‑autônoma com usuários ou outros sistemas.


#### DEBUGANDO COMO ESTA O STORE

In [29]:
for session_id, history in store.items():
    print(f"{{'{session_id}': InMemoryChatMessageHistory(messages=[")
    for msg in history.messages:
        print(f"  {msg!r},")
    print("])}")

{'usuario_mateus_123': InMemoryChatMessageHistory(messages=[
  HumanMessage(content='Boa tarde, meu nome é Mateus e estou estudando pra me tornar um engenheiro de agentes de ia', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Boa tarde, Mateus! 👋\n\nQue bacana saber que você está se preparando para se tornar um engenheiro de agentes de IA. Esse campo está crescendo rapidamente e oferece muitas oportunidades interessantes.\n\n### Como posso ajudar você hoje?\n\n1. **Fundamentos de IA e Machine Learning**  \n   - Conceitos básicos, algoritmos, métricas, etc.\n\n2. **Arquiteturas de Agentes**  \n   - Agentes reativos, deliberativos, híbridos, multi‑agentes, etc.\n\n3. **Frameworks e Ferramentas**  \n   - OpenAI, LangChain, Rasa, Microsoft Bot Framework, etc.\n\n4. **Desenvolvimento de Conversational Agents**  \n   - NLP, LLMs, fine‑tuning, prompt engineering, avaliação de diálogos.\n\n5. **Ética e Responsabilidade**  \n   - Bias, privacidade, segurança, governança de IA

##### DEBUGANDO OQUE É INJETADO AO DARMOS INVOKE()

In [33]:
from langchain_core.globals import set_debug


set_debug(True)


res = with_message_history.invoke(
    {"input": "Qual é o meu nome e o que eu faço?"},
    config={"configurable": {"session_id": "usuario_mateus_123"}}
)

set_debug(False)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "Qual é o meu nome e o que eu faço?"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:load_history] Entering Chain run with input:
{
  "input": "Qual é o meu nome e o que eu faço?"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:load_history] s] Exiting Chain run with output:
[outputs]
[chain/start] [chain:RunnableWithMessageHistory > chain:check_sync_or_async] Entering Chain run with input:
[inputs]
[llm/start] [chain:RunnableWithMessageHistory > chain:check_sync_or_async > llm:ChatGroq] Entering LLM run with input:
{
  "prompts": [
    "Human: Boa tarde, meu nome é Mateus e estou estudando pra me tornar um engenheiro de agentes de ia\nAI: Boa tarde, Mateus! 👋\n\nQue bacana saber que você está se preparando para se tornar um engenheiro de agentes de IA. Esse campo está crescendo rapidamente e oferece muitas oportunidades interessantes.\n\n### Como posso ajudar você hoje?\n\n1.